ENTSO-E Transparency has phased out the SFTP server. New approach can be found here:

https://transparencyplatform.zendesk.com/hc/en-us/articles/35960137882129-File-Library-Guide

For now, files were downloaded manually

In [1]:
import pysftp
import sys
import os
import pandas as pd

c:\Users\jonas\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2024'

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [5]:
dir_out = "../parsed_data/"

In [6]:
map_country = {
'AT' : 'AT',
'BE' : 'BE',
'BG' : 'BG',
'CH' : 'CH',
'CZ' : 'CZ',
'DE_LU' : 'DE',
'DK1' : 'DK',
'DK2' : 'DK',
'EE' : 'EE',
'ES' : 'ES',
'FI' : 'FI',
'FR' : 'FR',
'GR' : 'GR',
'HR' : 'HR',
'HU' : 'HU',
'IE_SEM' : 'IT',
'IT-Calabria' :'IT', 
'IT-CNORTH' : 'IT',
'IT-CSOUTH' : 'IT',
'IT-NORTH' : 'IT',
'IT-SACOAC' : 'IT',
'IT-SACODC' : 'IT',
'IT-Sardinia' :'IT', 
'IT-Sicily' : 'IT',
'IT-SOUTH' : 'IT',
'LT' : 'LT',
'LV' : 'LV',
'ME' : 'ME',
'MK' : 'MK',
'NL' : 'NL',
'NO1' : 'NO',
'NO2' : 'NO',
'NO2NSL' : 'NO',
'NO3' : 'NO',
'NO4' : 'NO',
'NO5' : 'NO',
'PL' : 'PL',
'PT' : 'PT',
'RO' : 'RO',
'RS' : 'RS',
'SE1' : 'SE',
'SE2' : 'SE',
'SE3' : 'SE',
'SE4' : 'SE',
'SI' : 'SI',
'SK' : 'SK'
 }


In [7]:
bz_countries = {'IT','SE','DK','NO'}

In [8]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [9]:
# show list of all available folders (uncomment last line if needed)
# relevant folder was renamed to EnergyPrices_12.1.D_r3
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        files = sftp.listdir('/TP_export/')   
        print(files)

## load data

In [10]:
#set paths and get file names
path_prices = path+'EnergyPrices_12.1.D_r3/'
path_prices_local = path_local+'prices/'
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_prices)
if download =="no": 
    files = os.listdir(path_prices_local)
if year != "":
    files = [i for i in files if year in i]

In [11]:
#download aggregated data
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_prices+file,path_prices_local+file)
            print('Successfully downloaded file '+file)

In [12]:
#combine files to one data frame
df_prices = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_prices_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime(UTC)")
    df_prices = pd.concat([df_prices,df_temp])
df_prices = df_prices.drop(['InstanceCode','ResolutionCode','AreaCode','AreaDisplayName','AreaTypeCode','ContractType','UpdateTime(UTC)'], axis=1)
#df_prices_quarterhourly = df_prices.copy()[df_prices.Sequence == "2"] 
df_prices = df_prices[df_prices['Sequence'].isin(['1', ' '])].copy()    #we only want hourly resolution
df_prices.head()

,MapCode,Sequence,Price[Currency/MWh],Currency
DateTime(UTC),,,,
2024-01-01 00:00:00,AT,1,0.01,EUR
2024-01-01 01:00:00,AT,1,0.02,EUR
2024-01-01 02:00:00,AT,1,0.00,EUR
2024-01-01 03:00:00,AT,1,-0.01,EUR
2024-01-01 04:00:00,AT,1,-0.01,EUR


In [13]:
#Currency is now all EUR, we drop it and rename columns
df_prices = df_prices.drop(columns = ['Currency','Sequence']).reset_index().rename(columns={'DateTime(UTC)':'date'}).set_index(['date','MapCode'])
df_prices.head()

,,Price[Currency/MWh]
date,MapCode,
2024-01-01 00:00:00,AT,0.01
2024-01-01 01:00:00,AT,0.02
2024-01-01 02:00:00,AT,0.00
2024-01-01 03:00:00,AT,-0.01
2024-01-01 04:00:00,AT,-0.01


In [14]:
df_prices_stack = pd.DataFrame(df_prices.stack()).rename(columns={0:'EUR_per_MWh'}).reset_index()
df_prices_stack['country'] = df_prices_stack.MapCode.map(map_country)
df_prices_stack.head()

,date,MapCode,level_2,EUR_per_MWh,country
0,2024-01-01 00:00:00,AT,Price[Currency/MWh],0.01,AT
1,2024-01-01 01:00:00,AT,Price[Currency/MWh],0.02,AT
2,2024-01-01 02:00:00,AT,Price[Currency/MWh],0.00,AT
3,2024-01-01 03:00:00,AT,Price[Currency/MWh],-0.01,AT
4,2024-01-01 04:00:00,AT,Price[Currency/MWh],-0.01,AT


now we also load load values for weighting in countries with multiple bidding zones

In [15]:
#set paths and get file names
path_load = path+'ActualTotalLoad_6.1.A/'
path_load_local = path_local+'load/'
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_load)
if download =="no": 
    files = os.listdir(path_load_local)
if year != "":
    files = [i for i in files if year in i]

In [19]:
#download aggregated load data (ActualTotalLoad)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_load+file,path_load_local+file)
            print('Successfully downloaded file '+file)

In [20]:
#combine files to one data frame
df_load = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_load_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime")
    df_load = pd.concat([df_load,df_temp])
df_load = df_load[df_load.AreaTypeCode == "BZN"].drop(["AreaTypeCode",'ResolutionCode','AreaName','UpdateTime','AreaCode'], axis=1).reset_index()
df_load = df_load.sort_values(by=['DateTime'])
df_load.info()

<class 'pandas.core.frame.DataFrame'>
Index: 726064 entries, 2296 to 724648
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DateTime        726064 non-null  datetime64[ns]
 1   MapCode         726064 non-null  object        
 2   TotalLoadValue  726064 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 22.2+ MB


In [21]:
#some values are reported quarter hourly so we have to resample to hourly values
df_load_pivot = df_load.pivot_table(index="DateTime",columns="MapCode",values='TotalLoadValue')
df_load_pivot_hourly = df_load_pivot.resample('h').mean()
df_load_hourly = pd.DataFrame(df_load_pivot_hourly.stack()).rename(columns={0:'TotalLoadValue'}).reset_index()
df_load_hourly['country'] = df_load_hourly.MapCode.map(map_country)
df_load_hourly = df_load_hourly[df_load_hourly.country.isin(bz_countries)]
df_load_hourly = df_load_hourly.rename(columns={'DateTime':'date'}).reset_index()
df_load_hourly.head()

,index,date,MapCode,TotalLoadValue,country
0,9,2024-01-01,DK1,2241.740,DK
1,10,2024-01-01,DK2,1462.710,DK
2,19,2024-01-01,IE_SEM,4076.365,IT
3,20,2024-01-01,IT-CNORTH,1727.000,IT
4,21,2024-01-01,IT-CSOUTH,4517.000,IT


In [22]:
df_load_country = df_load_hourly.groupby(['date','country']).sum().reset_index()
df_load_country.head()

,date,country,index,MapCode,TotalLoadValue
0,2024-01-01 00:00:00,DK,19,DK1DK2,3704.450
1,2024-01-01 00:00:00,IT,180,IE_SEMIT-CNORTHIT-CSOUTHIT-CalabriaIT-NORTHIT-...,23986.365
2,2024-01-01 00:00:00,NO,175,NO1NO2NO3NO4NO5,18363.750
3,2024-01-01 00:00:00,SE,174,SE1SE2SE3SE4,16763.000
4,2024-01-01 01:00:00,DK,117,DK1DK2,3638.330


In [23]:
df_load_hourly = df_load_hourly.drop(columns='country')

In [24]:
df_prices_temp = df_prices_stack.merge(df_load_hourly,on=['date','MapCode'],how='left')
df_prices_temp_2 = df_prices_temp.merge(df_load_country,on=['date','country'],how='left')
df_prices_temp_2['weight'] = df_prices_temp_2.TotalLoadValue_x / df_prices_temp_2.TotalLoadValue_y
df_prices_temp_2['weight'][~df_prices_temp_2.country.isin(bz_countries)] = df_prices_temp_2['weight'].fillna(1)
df_prices_temp_2['EUR_per_MWh'] = df_prices_temp_2['EUR_per_MWh'] * df_prices_temp_2['weight']
df_prices_temp_2.tail()

C:\Users\jonas\AppData\Local\Temp\ipykernel_32312\3176826473.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_prices_temp_2['weight'][~df_prices_temp_2.country.isin(bz_countries)] = df_prices_temp_2['weight'].fillna(1)
C:\Users\jonas\A

,date,MapCode_x,level_2,EUR_per_MWh,country,index_x,TotalLoadValue_x,index_y,MapCode_y,TotalLoadValue_y,weight
412921,2024-12-31 19:00:00,UA_IPS,Price[Currency/MWh],4980.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
412922,2024-12-31 20:00:00,UA_IPS,Price[Currency/MWh],4980.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
412923,2024-12-31 21:00:00,UA_IPS,Price[Currency/MWh],5500.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
412924,2024-12-31 22:00:00,UA_IPS,Price[Currency/MWh],3500.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
412925,2024-12-31 23:00:00,UA_IPS,Price[Currency/MWh],2900.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [25]:
df_prices_hourly_stack = df_prices_temp_2.groupby(['date','country']).sum().reset_index()
df_prices_hourly_stack.head()

,date,country,MapCode_x,level_2,EUR_per_MWh,index_x,TotalLoadValue_x,index_y,MapCode_y,TotalLoadValue_y,weight
0,2024-01-01,AT,AT,Price[Currency/MWh],0.01,0.0,0.0,0.0,0,0.0,1.0
1,2024-01-01,BE,BE,Price[Currency/MWh],0.01,0.0,0.0,0.0,0,0.0,1.0
2,2024-01-01,BG,BG,Price[Currency/MWh],0.01,0.0,0.0,0.0,0,0.0,1.0
3,2024-01-01,CH,CH,Price[Currency/MWh],21.99,0.0,0.0,0.0,0,0.0,1.0
4,2024-01-01,CZ,CZ,Price[Currency/MWh],0.01,0.0,0.0,0.0,0,0.0,1.0


In [26]:
df_prices_hourly = df_prices_hourly_stack[['date','country','EUR_per_MWh']].pivot_table(index='date',columns='country',values='EUR_per_MWh')
df_prices_hourly.head()

country,AT,BE,BG,CH,CZ,DE,DK,EE,ES,FI,...,MK,NL,NO,PL,PT,RO,RS,SE,SI,SK
date,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.01,0.01,0.01,21.99,0.01,0.01,28.266353,28.46,50.09,28.46,...,86.04,0.01,42.347332,74.50,50.09,0.01,5.02,28.46,0.01,0.01
2024-01-01 01:00:00,0.02,0.00,0.04,14.32,0.02,0.00,26.660000,26.66,47.50,26.66,...,57.44,0.00,30.046090,73.29,47.50,0.04,1.10,26.66,0.03,0.05
2024-01-01 02:00:00,0.00,-0.01,0.01,11.37,0.00,-0.01,11.857097,24.48,43.50,24.48,...,30.00,-0.01,27.549256,71.58,43.50,0.01,2.30,24.48,0.00,0.01
2024-01-01 03:00:00,-0.01,-0.03,0.01,11.35,-0.01,-0.03,9.083792,24.01,42.50,24.01,...,10.38,-0.03,27.037858,73.27,42.50,0.01,2.27,24.01,0.00,0.01
2024-01-01 04:00:00,-0.01,-0.02,0.01,11.39,-0.01,-0.02,5.986804,21.23,42.09,21.23,...,5.01,-0.02,25.018943,74.32,42.09,0.01,5.40,21.23,0.00,0.01


In [27]:
df_prices_hourly['LU']=df_prices_hourly['DE']

In [28]:
df_prices_hourly.to_csv(dir_out+'prices_'+year+'_hourly_entsoe.csv', encoding="utf-8")